# BM25 Search Engine: Validation and Production Scaling
This notebook demonstrates the mathematical properties of the BM25 scoring algorithm (Term Frequency Saturation and Document Length Normalization) using a controlled dataset, and then scales the engine to process a real-world dataset of 5,000 arXiv paper abstracts.

In [10]:
import pandas as pd
from search_engine import BM25SearchEngine

def display_results(query, results):
    """Función para imprimir resultados en tablas limpias"""
    df = pd.DataFrame(results, columns=["Doc ID", "Text", "BM25 Score"])
    display(df)

# Corpus pequeño de pruebas para validar la matemática
corpus_pruebas = {
    "doc_1": "Non-Archimedean dynamics studies the topological and algebraic properties of spaces over p-adic fields.",
    "doc_2": "Algebraic geometry is a branch of mathematics studying the zeros of multivariate polynomials.",
    "doc_short": "Search algorithms retrieve information.", 
    "doc_long": "Search algorithms are very useful. They retrieve information but also require a lot of computational resources and careful optimization to work well in production environments.", 
    "doc_spam": "Search search search search search search search algorithms." 
}

# Entrenamos el motor pequeño
engine_pruebas = BM25SearchEngine(k1=1.2, b=0.75)
engine_pruebas.fit(corpus_pruebas)

## 1. Document Length Normalization (The $b$ parameter)
A key feature of BM25 is penalizing longer documents so they don't unfairly outrank shorter, highly relevant ones. Both `doc_short` and `doc_long` contain the terms "search", "algorithms", "retrieve", and "information". Let's verify the engine correctly scores the denser document higher.

In [11]:
query = "search algorithms retrieve information"
print(f"Query: '{query}'")
results = engine_pruebas.search(query, top_n=3)
display_results(query, results)

Query: 'search algorithms retrieve information'


,Doc ID,Text,BM25 Score
0,doc_short,Search algorithms retrieve information.,3.9467
1,doc_long,Search algorithms are very useful. They retrie...,2.0535
2,doc_spam,Search search search search search search sear...,1.6965


## 2. Term Frequency Saturation (The $k_1$ parameter)

Unlike simple term-frequency models where repeating a word 7 times increases the score by 700%, BM25 uses the $k_1$ parameter to create an asymptotic limit (saturation). 

Notice how `doc_spam` repeats the word "search" 7 times, while `doc_short` only uses it once. Although `doc_spam` ranks first (as it logically should, given its density), its score is only marginally higher (approx 1.05 vs 0.75). The algorithm successfully prevents the score from scaling linearly, neutralizing keyword-stuffing tactics.

In [12]:
query = "search"
print(f"Query: '{query}'")
results = engine_pruebas.search(query, top_n=3)
display_results(query, results)

Query: 'search'


,Doc ID,Text,BM25 Score
0,doc_spam,Search search search search search search sear...,1.0569
1,doc_short,Search algorithms retrieve information.,0.7520
2,doc_long,Search algorithms are very useful. They retrie...,0.3913


## 3. Scale and Production: arXiv Dataset
Having mathematically validated the engine's edge cases, we now scale it to ingest and index 5,000 real scientific paper abstracts from arXiv.

In [13]:
print("Cargando dataset de arXiv...")
df = pd.read_csv("arxiv_data.csv").head(5000)

corpus_masivo = {}
for index, row in df.iterrows():
    doc_id = f"arxiv_{index}"
    texto = str(row['titles']) + " " + str(row['summaries'])
    corpus_masivo[doc_id] = texto.replace('\n', ' ')

print(f"Corpus masivo construido con {len(corpus_masivo)} artículos.")

# Entrenamos el motor masivo
engine_masivo = BM25SearchEngine(k1=1.2, b=0.75)
engine_masivo.fit(corpus_masivo)
print("Índice invertido masivo listo.")

Cargando dataset de arXiv...
Corpus masivo construido con 5000 artículos.
Índice invertido masivo listo.


## 4. Real-World Query Retrieval
Testing the scaled engine with a multi-term domain-specific query.

In [14]:
query_real = "algebraic geometry spaces"
print(f"Resultados masivos para: '{query_real}'\n")

resultados_masivos = engine_masivo.search(query_real, top_n=5)
display_results(query_real, resultados_masivos)

Resultados masivos para: 'algebraic geometry spaces'



,Doc ID,Text,BM25 Score
0,arxiv_3129,Beyond Vector Spaces: Compact Data Representat...,11.9035
1,arxiv_3387,Geometry of Deep Generative Models for Disenta...,11.4375
2,arxiv_2105,Switch Spaces: Learning Product Spaces with Sp...,11.0130
3,arxiv_10,Direct Estimation of Appearance Models for Seg...,9.9012
4,arxiv_3722,Learning Continuous Semantic Representations o...,9.8535
